# 📏 Model Evaluation Metrics — The Complete Guide

## Accuracy, Precision, Recall, F1-Score, Confusion Matrix, ROC-AUC

---

## 🎯 What Will You Learn?

- Understand **WHY accuracy alone is not enough**
- Learn each metric with intuitive, real-world examples
- Build intuition for **when to use which metric**
- Compare **all 5 models** (LR, KNN, DT, RF, SVM) using all metrics
- Understand the **Precision-Recall trade-off**

---

## 📖 Table of Contents
1. Why Accuracy Is Not Always Enough
2. The Confusion Matrix — The Foundation of Everything
3. Accuracy
4. Precision
5. Recall (Sensitivity)
6. F1-Score
7. ROC-AUC Curve
8. Precision-Recall Curve
9. Comparing All 5 Models
10. Summary: Which Metric to Use When?


---
## 1. ❓ Why Accuracy Is Not Always Enough

### The Medical Disaster Scenario 🏥

Imagine a hospital develops a COVID test:
- 95 patients are **healthy** (negative)
- 5 patients are **sick** (positive)

A lazy algorithm that **always says "Healthy" for everyone** would get:

```
Accuracy = 95/100 = 95% ← Sounds great!
```

But this model is **completely useless** — it never detects sick patients!

**Lesson:** Accuracy can be misleading, especially with **imbalanced data** (unequal class sizes).

### A Fraud Detection Example 💳

- 9,900 normal transactions  
- 100 fraudulent transactions

A model that calls everything "normal" → **Accuracy = 99%**, but catches **0 frauds!**

**We need better metrics.** Let's learn them all.


In [ ]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ All libraries imported!")

In [ ]:
# Load and prepare the Customer Churn dataset
df = pd.read_csv('data/customer_churn.csv')
df_model = df.drop(['customer_id'], axis=1)

# Handle missing values
numeric_cols = df_model.select_dtypes(include=[np.number]).columns
df_model[numeric_cols] = df_model[numeric_cols].fillna(df_model[numeric_cols].median())

# Encode categorical columns
le = LabelEncoder()
for col in df_model.select_dtypes(include=['object']).columns:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

X = df_model.drop('churned', axis=1)
y = df_model['churned']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Dataset: Customer Churn")
print(f"Training: {len(X_train)} | Testing: {len(X_test)}")
print(f"Class distribution in test set:")
print(f"  Stayed (0):  {(y_test==0).sum()} ({(y_test==0).mean()*100:.1f}%)")
print(f"  Churned (1): {(y_test==1).sum()} ({(y_test==1).mean()*100:.1f}%)")

---
## 2. 🔲 The Confusion Matrix — The Foundation of Everything

**All classification metrics are calculated from the Confusion Matrix!**

Let's understand it thoroughly.

```
                 │  Predicted NEGATIVE  │  Predicted POSITIVE  │
─────────────────┼──────────────────────┼──────────────────────┤
 Actual NEGATIVE │   True Negative (TN) │  False Positive (FP) │
─────────────────┼──────────────────────┼──────────────────────┤
 Actual POSITIVE │  False Negative (FN) │   True Positive (TP) │
─────────────────┴──────────────────────┴──────────────────────┘
```

In our case (Churn prediction):
- **Positive = Churn (1)**
- **Negative = Stay (0)**

| Cell | Meaning | Business Impact |
|------|---------|----------------|
| **TN** | Predicted Stay → Actually Stayed ✅ | Correct, no action needed |
| **FP** | Predicted Churn → Actually Stayed ❌ | Wasted retention offer |
| **FN** | Predicted Stay → Actually Churned ❌ | **MISSED a churner! Costly!** |
| **TP** | Predicted Churn → Actually Churned ✅ | Caught it! Offer discount |

**For churn prediction, FN is the most dangerous error** — you missed a customer who left!

In [ ]:
# Train Logistic Regression as our example model
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)
y_pred = lr.predict(X_test_s)
y_prob = lr.predict_proba(X_test_s)

# The Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

print("Confusion Matrix for Logistic Regression:")
print(f"\n                     Predicted STAY  Predicted CHURN")
print(f"  Actual STAY         {TN:5d} (TN)       {FP:5d} (FP)")
print(f"  Actual CHURN        {FN:5d} (FN)       {TP:5d} (TP)")
print(f"\n  Total test samples: {len(y_test)}")
print(f"  Correctly classified: {TN + TP} ({(TN+TP)/len(y_test)*100:.1f}%)")
print(f"  Misclassified:        {FP + FN} ({(FP+FN)/len(y_test)*100:.1f}%)")

In [ ]:
# Visualize the confusion matrix beautifully
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Standard confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Predicted: Stay', 'Predicted: Churn'],
            yticklabels=['Actual: Stay', 'Actual: Churn'],
            linewidths=3, linecolor='white',
            annot_kws={'size': 20, 'weight': 'bold'})
axes[0].set_title('Confusion Matrix\n(Raw Counts)', fontsize=14, fontweight='bold')

# Normalized confusion matrix (percentages)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=['Predicted: Stay', 'Predicted: Churn'],
            yticklabels=['Actual: Stay', 'Actual: Churn'],
            linewidths=3, linecolor='white',
            annot_kws={'size': 16, 'weight': 'bold'})
axes[1].set_title('Normalized Confusion Matrix\n(Row Percentages)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrix_explained.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"\nReading: Of all customers who ACTUALLY churned, we caught {cm_norm[1,1]:.1%} of them")

---
## 3. 🎯 Accuracy

**Formula:** `Accuracy = (TP + TN) / Total`

**English:** "Out of all predictions, what fraction was correct?"

```
Accuracy = (True Positive + True Negative) / Total Predictions
         = (Correct predictions) / (All predictions)
```

**Analogy:** Your exam score — "I got 85 out of 100 questions right" = 85% accuracy.

**When to use:** When classes are balanced AND both types of errors are equally costly.

**When NOT to use:** Imbalanced data (fraud detection, disease diagnosis, etc.)

In [ ]:
# Calculate Accuracy step by step
accuracy_manual = (TP + TN) / (TP + TN + FP + FN)
accuracy_sklearn = accuracy_score(y_test, y_pred)

print("Accuracy Calculation:")
print(f"  TP = {TP}, TN = {TN}, FP = {FP}, FN = {FN}")
print(f"  Accuracy = (TP + TN) / (TP + TN + FP + FN)")
print(f"           = ({TP} + {TN}) / ({TP} + {TN} + {FP} + {FN})")
print(f"           = {TP+TN} / {TP+TN+FP+FN}")
print(f"           = {accuracy_manual:.4f}")
print(f"           = {accuracy_manual*100:.2f}%")
print(f"\n  Sklearn confirms: {accuracy_sklearn:.4f} ✅")

---
## 4. 🎯 Precision

**Formula:** `Precision = TP / (TP + FP)`

**English:** "Of all customers I PREDICTED would churn, how many ACTUALLY did?"

```
Precision = TP / (TP + FP)
           = True Positives / (True Positives + False Positives)
```

**Analogy:** A fisherman using a net:
- Precision = "Of all fish I caught in my net, what % were the fish I wanted?"
- High Precision = very few rocks and seaweed in the net

**When to care about Precision:**
- Spam filter: Don't want to mark real emails as spam (FP is costly)
- Legal decisions: Don't want to wrongly accuse innocent people
- Sending retention offers: Don't waste money calling people who wouldn't churn anyway

In [ ]:
# Calculate Precision
precision_manual = TP / (TP + FP)
precision_sklearn = precision_score(y_test, y_pred)

print("Precision Calculation:")
print(f"  TP = {TP}, FP = {FP}")
print(f"  Precision = TP / (TP + FP)")
print(f"            = {TP} / ({TP} + {FP})")
print(f"            = {TP} / {TP+FP}")
print(f"            = {precision_manual:.4f} = {precision_manual*100:.2f}%")
print()
print(f"  Interpretation: When our model predicted a customer would churn,")
print(f"  it was correct {precision_manual*100:.1f}% of the time.")
print(f"  The other {(1-precision_manual)*100:.1f}% were false alarms (stayed but we predicted churn).")

---
## 5. 🔎 Recall (Sensitivity / True Positive Rate)

**Formula:** `Recall = TP / (TP + FN)`

**English:** "Of all customers who ACTUALLY churned, how many did we CATCH?"

```
Recall = TP / (TP + FN)
       = True Positives / (True Positives + False Negatives)
       = Actual positives that we caught
```

**Analogy:** A smoke detector:
- Recall = "Of all actual fires, what % did my detector catch?"
- High Recall = detector catches almost every fire
- Low Recall = many fires go undetected → catastrophic!

**When to care about Recall:**
- Cancer detection: Must catch EVERY possible case (FN is catastrophic)
- Fraud detection: Must catch EVERY fraud
- Churn prediction: Must catch EVERY customer at risk

**The Trade-off:**
You can always increase recall to 100% by predicting EVERYONE as positive!
But then precision collapses → you'd call every customer every day!

In [ ]:
# Calculate Recall
recall_manual = TP / (TP + FN)
recall_sklearn = recall_score(y_test, y_pred)

print("Recall Calculation:")
print(f"  TP = {TP}, FN = {FN}")
print(f"  Recall = TP / (TP + FN)")
print(f"         = {TP} / ({TP} + {FN})")
print(f"         = {TP} / {TP+FN}")
print(f"         = {recall_manual:.4f} = {recall_manual*100:.2f}%")
print()
print(f"  Interpretation: Of all customers who ACTUALLY churned,")
print(f"  our model caught {recall_manual*100:.1f}% of them.")
print(f"  We MISSED {(1-recall_manual)*100:.1f}% — those are the dangerous False Negatives!")

In [ ]:
# Visual illustration of Precision vs Recall
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Create circles to represent populations
from matplotlib.patches import FancyBboxPatch

# Precision visualization
ax = axes[0]
total_predicted_pos = TP + FP
circle_data_p = [('TP (Correct Churn Catches)', TP, '#2ecc71'), 
                 ('FP (False Alarms)', FP, '#e74c3c')]
wedge_sizes = [TP, FP]
wedge_colors = ['#2ecc71', '#e74c3c']
labels_p = [f'TP = {TP}\n(Correctly predicted churn)', 
            f'FP = {FP}\n(Wrongly predicted churn)']
ax.pie(wedge_sizes, labels=labels_p, colors=wedge_colors, autopct='%1.1f%%',
       startangle=90, textprops={'fontsize': 10})
ax.set_title(f'PRECISION = {precision_manual*100:.1f}%\n"Of all predicted churners, what % actually churned?"',
             fontsize=12, fontweight='bold')

# Recall visualization
ax = axes[1]
wedge_sizes_r = [TP, FN]
wedge_colors_r = ['#2ecc71', '#e74c3c']
labels_r = [f'TP = {TP}\n(Churners we caught)', 
            f'FN = {FN}\n(Churners we MISSED!)']
ax.pie(wedge_sizes_r, labels=labels_r, colors=wedge_colors_r, autopct='%1.1f%%',
       startangle=90, textprops={'fontsize': 10})
ax.set_title(f'RECALL = {recall_manual*100:.1f}%\n"Of all actual churners, what % did we catch?"',
             fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('precision_recall_visual.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 6. ⚖️ F1-Score — The Balanced Metric

**Formula:** `F1 = 2 × (Precision × Recall) / (Precision + Recall)`

**English:** "The harmonic mean of Precision and Recall — a single number that balances both."

```
F1 = 2 × Precision × Recall
         ──────────────────
         Precision + Recall
```

**Why harmonic mean (not regular average)?**
- Regular average: (1.0 + 0.0) / 2 = 0.50 ← seems decent!
- Harmonic mean: 2×(1.0×0.0)/(1.0+0.0) = 0.0 ← correctly shows it's terrible!

The harmonic mean **penalizes extreme values** — if either precision or recall is 0, F1 = 0.

**Analogy:** Like a student who needs BOTH to attend class AND pass exams:
- Attends class 100% but passes 0% exams → useless student!
- F1 would be 0 here

**F1 score ranges:**
- **1.0** = Perfect (both precision and recall are 100%)
- **0.5** = Poor
- **0.0** = Completely wrong on one or both metrics

In [ ]:
# Calculate F1-Score
f1_manual = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual)
f1_sklearn = f1_score(y_test, y_pred)

print("F1-Score Calculation:")
print(f"  Precision = {precision_manual:.4f}")
print(f"  Recall    = {recall_manual:.4f}")
print(f"  F1 = 2 × (P × R) / (P + R)")
print(f"     = 2 × ({precision_manual:.4f} × {recall_manual:.4f}) / ({precision_manual:.4f} + {recall_manual:.4f})")
print(f"     = {2*precision_manual*recall_manual:.4f} / {precision_manual+recall_manual:.4f}")
print(f"     = {f1_manual:.4f}")
print()
print("💡 F1 Summary:")
print(f"   Accuracy:  {accuracy_sklearn:.4f} = {accuracy_sklearn*100:.2f}%")
print(f"   Precision: {precision_sklearn:.4f} = {precision_sklearn*100:.2f}%")
print(f"   Recall:    {recall_sklearn:.4f} = {recall_sklearn*100:.2f}%")
print(f"   F1-Score:  {f1_sklearn:.4f} = {f1_sklearn*100:.2f}%")

In [ ]:
# Visualize the relationship between precision, recall, and F1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metrics bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [accuracy_sklearn, precision_sklearn, recall_sklearn, f1_sklearn]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
bars = axes[0].bar(metrics, [v*100 for v in values], color=colors, edgecolor='black', width=0.6)
axes[0].set_ylim([0, 110])
axes[0].set_ylabel('Score (%)', fontsize=12)
axes[0].set_title('All Metrics — Logistic Regression\n(Customer Churn)', fontsize=13, fontweight='bold')
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val*100:.1f}%', ha='center', fontsize=12, fontweight='bold')

# Precision-Recall trade-off curve
prec_vals, rec_vals, thresh = precision_recall_curve(y_test, y_prob[:, 1])
ap = average_precision_score(y_test, y_prob[:, 1])

axes[1].plot(rec_vals, prec_vals, lw=2.5, color='#9b59b6',
             label=f'Logistic Regression (AP={ap:.3f})')
axes[1].scatter([recall_sklearn], [precision_sklearn], s=200, color='#e74c3c',
                zorder=5, label=f'Current threshold\n(P={precision_sklearn:.2f}, R={recall_sklearn:.2f})')
axes[1].axhline(y=y_test.mean(), color='gray', linestyle='--', lw=1.5, 
                label=f'Baseline ({y_test.mean():.2f})')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curve\n(Higher = Better)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('all_metrics.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 7. 📈 ROC-AUC — Receiver Operating Characteristic

### What is the ROC Curve?

The ROC curve shows **how well a model separates the two classes** across all possible threshold values.

**X-axis:** False Positive Rate (FPR) = FP / (FP + TN) → "How often do we wrong alarm?"
**Y-axis:** True Positive Rate (TPR) = TP / (TP + FN) = Recall → "How many real cases do we catch?"

### Threshold? What threshold?

Our model doesn't just output 0 or 1 — it outputs a probability like 0.73.
We need a **threshold** to convert probability → prediction:
- Threshold = 0.5: If prob > 0.5 → predict Churn
- Threshold = 0.3: If prob > 0.3 → predict Churn (catches more, but more false alarms)
- Threshold = 0.8: If prob > 0.8 → predict Churn (conservative, misses more)

The ROC curve plots all possible thresholds at once!

### AUC (Area Under the Curve)

```
AUC = 1.0 → Perfect model (top-left corner)
AUC = 0.9 → Excellent
AUC = 0.8 → Good
AUC = 0.7 → Fair
AUC = 0.5 → Random guessing (the diagonal line)
AUC < 0.5 → Worse than random!
```

**Analogy:** AUC measures the probability that a randomly chosen positive sample has a higher predicted probability than a randomly chosen negative sample.

In [ ]:
# Demonstrate the effect of different thresholds
print("Effect of different thresholds on Precision, Recall, and F1:")
print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<10} {'Accuracy'}")
print("-" * 60)

for thresh in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    y_pred_t = (y_prob[:, 1] >= thresh).astype(int)
    if y_pred_t.sum() > 0:  # Avoid division by zero
        p = precision_score(y_test, y_pred_t, zero_division=0)
        r = recall_score(y_test, y_pred_t, zero_division=0)
        f = f1_score(y_test, y_pred_t, zero_division=0)
        a = accuracy_score(y_test, y_pred_t)
        marker = " ← default" if thresh == 0.5 else ""
        print(f"{thresh:<12.1f} {p:<12.4f} {r:<12.4f} {f:<10.4f} {a:.4f}{marker}")

In [ ]:
# Beautiful ROC curve with threshold markers
fpr_vals, tpr_vals, thresholds = roc_curve(y_test, y_prob[:, 1])
auc_score = roc_auc_score(y_test, y_prob[:, 1])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ROC Curve
axes[0].plot(fpr_vals, tpr_vals, lw=3, color='#e74c3c', label=f'LR (AUC = {auc_score:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random guessing (AUC=0.5)')

# Mark some key thresholds
key_thresholds = [0.3, 0.5, 0.7]
for t in key_thresholds:
    idx = np.argmin(np.abs(thresholds - t))
    axes[0].scatter(fpr_vals[idx], tpr_vals[idx], s=150, zorder=5)
    axes[0].annotate(f'  t={t}', (fpr_vals[idx], tpr_vals[idx]), fontsize=9)

axes[0].fill_between(fpr_vals, tpr_vals, alpha=0.1, color='#e74c3c', label='AUC area')
axes[0].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
axes[0].set_ylabel('True Positive Rate (Recall)', fontsize=12)
axes[0].set_title(f'ROC Curve\nAUC = {auc_score:.3f}', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.02])

# What does AUC mean visually
ax = axes[1]
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

auc_models = [
    ("Perfect Model", 1.0, '#27ae60'),
    ("Excellent (AUC=0.9)", 0.9, '#2ecc71'),
    ("Our Model", auc_score, '#e74c3c'),
    ("Good (AUC=0.75)", 0.75, '#f39c12'),
    ("Random (AUC=0.5)", 0.5, 'gray')
]

# Approximate ROC curves for different AUC values
x_range = np.linspace(0, 1, 100)
for name, auc_val, color in auc_models:
    if auc_val == 1.0:
        y_range = np.where(x_range < 0.001, 1.0, 1.0)
        ax.plot([0, 0, 1], [0, 1, 1], lw=2, color=color, label=f'{name} (AUC={auc_val})')
    elif auc_val == 0.5:
        ax.plot([0, 1], [0, 1], '--', lw=2, color=color, label=f'{name} (AUC={auc_val})')
    else:
        y_range = x_range ** (1 / (2*auc_val - 0.5 + 0.001))
        ax.plot(x_range, y_range, lw=2, color=color, label=f'{name} (AUC≈{auc_val})')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('What Different AUC Values Look Like', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('roc_auc_explained.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 8. 🏆 Comparing All 5 Models on All Metrics

Now let's train all 5 models we've learned and compare them side by side!

In [ ]:
# Train all 5 models
print("Training all 5 models... (please wait)")

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=9),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM (RBF)': SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)
}

results = {}

for name, model in models.items():
    # Decision Tree and Random Forest don't need scaling
    if name in ['Decision Tree', 'Random Forest']:
        model.fit(X_train, y_train)
        y_pred_m = model.predict(X_test)
        y_prob_m = model.predict_proba(X_test)[:, 1]
    else:
        model.fit(X_train_s, y_train)
        y_pred_m = model.predict(X_test_s)
        y_prob_m = model.predict_proba(X_test_s)[:, 1]
    
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred_m),
        'Precision': precision_score(y_test, y_pred_m, zero_division=0),
        'Recall': recall_score(y_test, y_pred_m, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred_m, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob_m),
        'y_pred': y_pred_m,
        'y_prob': y_prob_m
    }
    print(f"  ✅ {name} trained")

print("\nAll models trained!")

In [ ]:
# Create comparison DataFrame
metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
comparison_df = pd.DataFrame(
    {name: {m: results[name][m] for m in metrics_cols} 
     for name in results}
).T.round(4)

print("Complete Model Comparison (Customer Churn Dataset):")
print("=" * 80)
print(comparison_df.to_string())
print()

# Find the best model for each metric
print("🏆 Best model per metric:")
for metric in metrics_cols:
    best = comparison_df[metric].idxmax()
    val = comparison_df[metric].max()
    print(f"   {metric:<15}: {best:<25} ({val:.4f})")

In [ ]:
# Comprehensive visualization of all metrics
fig = plt.figure(figsize=(20, 12))

model_names_short = ['LR', 'KNN', 'DT', 'RF', 'SVM']
full_names = list(results.keys())
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12']

# 1. Grouped bar chart — all metrics
ax1 = plt.subplot(2, 3, (1, 2))
x = np.arange(len(metrics_cols))
width = 0.15
for i, (name, short, color) in enumerate(zip(full_names, model_names_short, colors)):
    vals = [results[name][m] for m in metrics_cols]
    ax1.bar(x + i*width - 2*width, vals, width, label=f'{short} ({name})', 
            color=color, edgecolor='black', alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(metrics_cols, fontsize=11)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_ylim([0.5, 1.05])
ax1.set_title('All Models — All Metrics Comparison', fontsize=14, fontweight='bold')
ax1.legend(fontsize=9, loc='lower right')
ax1.axhline(y=0.8, color='red', linestyle=':', alpha=0.5, label='0.8 threshold')

# 2. ROC curves for all models
ax2 = plt.subplot(2, 3, 3)
for name, short, color in zip(full_names, model_names_short, colors):
    fpr, tpr, _ = roc_curve(y_test, results[name]['y_prob'])
    auc = results[name]['ROC-AUC']
    ax2.plot(fpr, tpr, lw=2, color=color, label=f'{short} (AUC={auc:.3f})')
ax2.plot([0, 1], [0, 1], 'k--', lw=1)
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
ax2.legend(fontsize=9, loc='lower right')

# 3-7. Confusion matrices for all models
for idx, (name, short, color) in enumerate(zip(full_names, model_names_short, colors)):
    ax = plt.subplot(2, 3, 4 + idx if idx < 3 else None)
    if idx < 3:
        cm_i = confusion_matrix(y_test, results[name]['y_pred'])
        sns.heatmap(cm_i, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Stay', 'Churn'],
                    yticklabels=['Stay', 'Churn'],
                    linewidths=2, linecolor='white',
                    annot_kws={'size': 13, 'weight': 'bold'})
        acc_i = results[name]['Accuracy']
        ax.set_title(f'{name}\nAcc={acc_i*100:.1f}%', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('all_models_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Radar/Spider chart — visual comparison
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(polar=True))

metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
N = len(metric_names)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_names, size=12)
ax.set_ylim(0.5, 1.0)
ax.set_yticks([0.6, 0.7, 0.8, 0.9, 1.0])
ax.set_yticklabels(['0.6', '0.7', '0.8', '0.9', '1.0'], size=9)

for name, short, color in zip(full_names, model_names_short, colors):
    values = [results[name][m] for m in metric_names]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=name)
    ax.fill(angles, values, alpha=0.08, color=color)

ax.set_title('Model Performance Comparison\n(Larger = Better)', 
             fontsize=15, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=10)

plt.tight_layout()
plt.savefig('radar_chart.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 9. 📚 When to Use Which Metric?

### The Decision Guide

```
┌─────────────────────────────────────────────────────────┐
│             WHICH METRIC SHOULD I USE?                  │
├─────────────────────────────────────────────────────────┤
│ Classes are balanced (roughly equal)?                   │
│   → Use ACCURACY or F1-Score                            │
├─────────────────────────────────────────────────────────┤
│ Classes are IMBALANCED (one much rarer)?                │
│   → Never use accuracy alone!                           │
│   → Use F1-Score, Precision, Recall, ROC-AUC            │
├─────────────────────────────────────────────────────────┤
│ Missing a positive is CATASTROPHIC?                     │
│   (Disease detection, fraud, churn)                     │
│   → Maximize RECALL                                     │
├─────────────────────────────────────────────────────────┤
│ False alarms are COSTLY?                                │
│   (Spam filter, unnecessary tests, legal decisions)     │
│   → Maximize PRECISION                                  │
├─────────────────────────────────────────────────────────┤
│ Need balance between both?                              │
│   → Use F1-SCORE                                        │
├─────────────────────────────────────────────────────────┤
│ Comparing models across all thresholds?                 │
│   → Use ROC-AUC                                         │
└─────────────────────────────────────────────────────────┘
```

In [ ]:
# Real-world examples with recommended metrics
examples = [
    ("Cancer Detection",     "High",   "Low",   "RECALL",    "Must catch every cancer case!"),
    ("Spam Filter",          "Low",    "High",  "PRECISION", "Don't block real emails!"),
    ("Customer Churn",       "Medium", "Medium","F1-Score",  "Balance catching churners and false alarms"),
    ("Fraud Detection",      "High",   "Medium","RECALL+AUC","Can't miss fraud, some false alarms OK"),
    ("Image Classification","Low",    "Low",   "ACCURACY",  "Classes usually balanced"),
    ("Loan Approval",        "Low",    "High",  "PRECISION", "Don't approve risky loans!")
]

df_examples = pd.DataFrame(examples, columns=[
    'Use Case', 'Cost of FN', 'Cost of FP', 'Recommended Metric', 'Reason'
])

print("Real-World Metric Selection Guide:")
print("=" * 100)
print(df_examples.to_string(index=False))

In [ ]:
# Final comprehensive table
print("\n" + "=" * 80)
print("           FINAL SUMMARY: ALL METRICS EXPLAINED")
print("=" * 80)

summary = [
    ("Accuracy",  "(TP+TN)/Total",        "Overall correctness",              "Balanced classes"),
    ("Precision", "TP/(TP+FP)",           "Quality of positive predictions",  "False alarms costly"),
    ("Recall",    "TP/(TP+FN)",           "Coverage of actual positives",     "Missing positives costly"),
    ("F1-Score",  "2*P*R/(P+R)",          "Balance of Precision & Recall",   "Imbalanced + need balance"),
    ("ROC-AUC",   "Area under ROC curve",  "Overall discriminative ability",  "Comparing models")
]

print(f"{'Metric':<12} {'Formula':<25} {'Measures':<35} {'Use When'}")
print("-" * 80)
for row in summary:
    print(f"{row[0]:<12} {row[1]:<25} {row[2]:<35} {row[3]}")

In [ ]:
# Final best model
best_model_f1 = comparison_df['F1-Score'].idxmax()
best_model_auc = comparison_df['ROC-AUC'].idxmax()

print("=" * 65)
print("      COMPLETE COURSE SUMMARY")
print("=" * 65)
print()
print("Models Covered:")
print("  1. Logistic Regression  → Customer Churn (Notebook 07)")
print("  2. K-Nearest Neighbors  → Customer Churn (Notebook 08)")
print("  3. Decision Tree        → Heart Disease  (Notebook 09)")
print("  4. Random Forest        → Wine Quality   (Notebook 10)")
print("  5. SVM                  → Wine Quality   (Notebook 11)")
print()
print("On Customer Churn Dataset:")
for name in full_names:
    acc = results[name]['Accuracy']
    f1 = results[name]['F1-Score']
    auc = results[name]['ROC-AUC']
    print(f"  {name:<25}: Acc={acc:.3f}, F1={f1:.3f}, AUC={auc:.3f}")
print()
print(f"  Best F1-Score:  {best_model_f1}")
print(f"  Best ROC-AUC:   {best_model_auc}")
print()
print("=" * 65)
print("🎓 Congratulations! You've completed the ML course!")
print("=" * 65)